<a href="https://colab.research.google.com/github/francji1/01NAEX/blob/main/projects/01NAEX_Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01NAEX - Team Project 1: The pendulum experiment

**Team:** *names of the team members and their roles (operator, timekeeper, data, analysis, ...)*

**Deadline: Sunday 8 November 2026, 23:59.** Submission rules are at the end of this notebook.

In this project you design, run and analyse your own experiment. The physics is simple and well known, which is exactly the
point: you can check whether your design and your statistical analysis recover the truth. The difficulty lies in the
**precision**: some of the effects you are asked to detect are only a few hundredths of a second.

## Background

For small release angles the period of a simple pendulum of length $L$ is
$$
T_0 = 2\pi\sqrt{\frac{L}{g}},
$$
independent of the mass of the bob. For a release angle $\theta_0$ (in radians) the period is longer,
$$
T(\theta_0) \approx T_0\left(1+\frac{\theta_0^2}{16}+\frac{11\,\theta_0^4}{3072}\right),
\qquad \log T = \log(2\pi) - \tfrac12\log g + \tfrac12 \log L + \log\left(1+\frac{\theta_0^2}{16}+\frac{11\,\theta_0^4}{3072}\right).
$$

| $\theta_0$ | 10° | 20° | 30° | 45° | 50° | 60° |
|---|---|---|---|---|---|---|
| $T/T_0 - 1$ | 0.19 % | 0.77 % | 1.74 % | 4.0 % | 5.0 % | 7.3 % |

For $L = 0.6$ m ($T_0 = 1.55$ s) the period at 30° is longer by only 0.024 s than at 10°, at 50° by 0.074 s.

Things that make a real pendulum differ from the ideal one:

* $L$ is the distance from the pivot to the **centre of mass** of the bob, not to the point where the string is tied.
  A bob of radius $R$ adds about $2R^2/(5L)$ to the effective length.
* The amplitude decreases during the timed oscillations (air drag), more for a light bob with a large surface.
* The pivot must not move; the string must not stretch or twist.
* A hand-operated stopwatch has a reaction-time error of roughly 0.05-0.2 s at the start and at the stop. Timing $N$ oscillations
  divides this error by $N$.

Last year most teams used angles of 5-20° and reported that "the angle has no effect" and that "mass is not significant,
therefore it does not matter". Neither conclusion followed from their data: the angle effect was too small for their precision,
and a non-significant test is not evidence of no effect.

## Research questions

1. How does the period depend on the length? Estimate the exponent of $L$ (theory: 1/2) and the gravitational acceleration $g$
   with a confidence interval.
2. Is the large-angle effect detectable, and does its size agree with the theory (the formula for $T(\theta_0)$ above)?
3. Can you show that the mass of the bob has **no practically relevant** effect on the period?

## Design requirements

These are minimum requirements; you may do more.

* **Factors** (full factorial):
  * length $L$: 3 levels across your feasible range (e.g. 0.3-1.0 m), measured from the pivot to the centre of mass of the bob;
  * release angle $\theta_0$: 3 levels, at least one $\le 10°$ and at least one $\ge 45°$ (set the angle by a protractor or by
    the horizontal displacement $d$, $\theta_0 = \arcsin(d/L)$);
  * mass $m$: 2 levels with the **same size and shape** of the bob (e.g. the same closed container empty and filled with sand or
    coins); the heavier one should be at least three times the lighter. Weigh both.
* **Response:** the period $T = t_N/N$ from the time $t_N$ of $N \ge 10$ full oscillations. Start timing at a pass through the
  lowest point after the pendulum has settled, not at the release. Record the raw $t_N$, not only $T$.
* **Pilot study:** before the main experiment, time at least 10 runs at one setting (e.g. the middle length, 10°, light bob)
  to estimate the run-to-run standard deviation $\sigma$ of $T$.
* **Sample size:** from the pilot estimate, choose the number of replicates so that the power of the $F$-test for the angle
  main effect is at least 0.9 for the difference between 10° and 30° (Lecture 3: noncentral $F$, least favourable configuration).
  You need at least 2 replicates of the $3 \times 3 \times 2 = 18$ runs.
* **Blocking:** each complete replicate of the 18 runs is a **block** (one operator, or one session). If several team members
  time the runs, the operator is a natural block (Lecture 3, radar example).
* **Randomization:** randomize the run order within each block in Python with a fixed seed and save the run sheet
  (Exercise 04, Section 4). Changing $L$ is laborious; if you decide to group the runs by length, say so before the experiment
  and discuss what this restriction on randomization means for the analysis (this design has a name - we will meet it in
  Lecture 11).
* **Record** for every run: run number, block, operator, date and time, $L$, $\theta_0$, mass, $N$, $t_N$ and a note on anything
  unusual (missed count, the bob hit something, ...). Do not delete runs; mark them and explain.

## Helper functions

You may use these functions (or write your own).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf


def anova_power(lam, df1, df2, alpha=0.05):
    """Power of the ANOVA F-test: P(F'(df1, df2, lam) > F_{1-alpha; df1, df2})."""
    return stats.ncf.sf(stats.f.ppf(1 - alpha, df1, df2), df1, df2, lam)


def tukey_from_model(model, data, factor, response, alpha=0.05):
    """Tukey's HSD for the levels of `factor`, using MS_E and df_E of a fitted model (balanced data)."""
    mse, dfe = model.mse_resid, model.df_resid
    means = data.groupby(factor, observed=True)[response].mean()
    n = data.groupby(factor, observed=True)[response].count().iloc[0]
    k = len(means)
    se = np.sqrt(mse / n)
    hsd = stats.studentized_range.ppf(1 - alpha, k, dfe) * se
    rows = []
    levels = list(means.index)
    for i in range(k):
        for j in range(i + 1, k):
            diff = means.iloc[j] - means.iloc[i]
            rows.append({"pair": f"{levels[j]} - {levels[i]}", "diff": diff,
                         "lower": diff - hsd, "upper": diff + hsd,
                         "p_adj": stats.studentized_range.sf(abs(diff) / se, k, dfe),
                         "reject": abs(diff) > hsd})
    print(f"MS_E = {mse:.4g}, df_E = {dfe:.0f}, n per level = {n}, HSD = {hsd:.4g}")
    return pd.DataFrame(rows)


def theory_period(L, theta_deg, g=9.81):
    """Period of a simple pendulum of length L (m) released at theta_deg degrees (series to the 4th order)."""
    th = np.radians(theta_deg)
    return 2 * np.pi * np.sqrt(L / g) * (1 + th**2 / 16 + 11 * th**4 / 3072)

## Tasks

Answer every task in the cells below it: code with outputs **and** a short comment in words. For every test state the
hypotheses, the test statistic, the $p$-value and your decision.

### 1. Experimental setup

Describe the pendulum (pivot, string, bobs and their masses and sizes), how you measure $L$ and $\theta_0$, the timing method
and $N$. A photo of the setup is welcome (link it). Name the nuisance factors you expect and how you deal with each of them
(block, randomize, keep constant).

### 2. Pilot study and sample size

* Report the pilot data, $\bar T$ and $\hat\sigma$ of $T$, and compare $\bar T$ with `theory_period()`.
* Using $\hat\sigma$, compute the power of the test for the angle main effect for 2, 3, 4, ... replicates (blocks); the target
  difference is the theoretical difference between 10° and 30° at your middle length. Choose the number of replicates.
* What is the smallest mass effect (in seconds) your design can detect with power 0.9?
* State your final plan (levels, number of blocks, $N$) **before** you run the main experiment.

### 3. Randomized run sheet

Generate the run sheet in Python with a fixed seed (random order within each block) and show it. Keep the sheet in the
data file (column `run`).

### 4. Data and descriptive analysis

* Load your CSV file (see Submission) and check it: number of runs per cell and block, missing values, notes.
* Table of means and standard deviations of $T$ by $L$, $\theta_0$ and mass.
* Plots: $T$ versus $\theta_0$ for each $L$ (interaction plot), $T$ by mass, $T$ by block, and $T$ in run order.

### 5. ANOVA: a factorial in blocks

* Fit the factorial model with blocks, $T \sim \text{block} + L \times \theta_0 \times m$ (all factors categorical), and reduce
  it sensibly. Interpret the significant effects.
* Fit the same model to $\log T$. The physical model is multiplicative in $L$ and $\theta_0$: explain why the $L \times \theta_0$
  interaction can be significant for $T$ and disappear for $\log T$. Which scale do you prefer and why?
* Was blocking useful? Compute the relative efficiency of the blocked design (Lecture 3).

### 6. Which angles differ?

Compare the angle levels with Tukey's HSD using $MS_E$ and $df_E$ of your model (not `pairwise_tukeyhsd()`). If the angle
effect depends on $L$, compare the angles at each length (simple effects). Compare the estimated angle effects with the
theory (`theory_period()`).

### 7. Does mass matter? Equivalence, not significance

* Estimate the mass effect (heavy - light) with a 95 % confidence interval from your model.
* Choose an equivalence margin $\Delta$ (in seconds or in % of $T$) **and justify it** (e.g. the precision needed to estimate $g$
  to 1 %). Test $H_0: |\text{effect}| \ge \Delta$ against $H_1: |\text{effect}| < \Delta$ by two one-sided $t$-tests (TOST) with
  $\alpha = 0.05$; equivalently, check whether the 90 % confidence interval lies inside $(-\Delta, \Delta)$.
* Explain the difference between "the mass effect is not significant" and "the mass effect is negligible".

### 8. Model diagnostics

Residuals versus fitted values, normal Q-Q plot, residuals versus each factor, versus block (operator) and **versus run order**
(time trends, fatigue, a stretching string). Comment on any problems and on what you would change.

### 9. Physical model and the estimate of $g$

* Let $x(\theta_0) = \log\left(1+\theta_0^2/16+11\,\theta_0^4/3072\right)$ ($\theta_0$ in radians). At angles of 45° and more the
  $\theta_0^4$ term is not negligible, so a plain $\theta_0^2$ regressor would give a biased slope.
* Fit the regression $\log T = \beta_0 + \beta_1 \log L + \beta_2\, x(\theta_0) + \beta_3\,\text{mass} + \text{block}$.
  Test $\beta_1 = 1/2$ and $\beta_2 = 1$.
* With $\beta_1 = 1/2$ and $\beta_2 = 1$ fixed (fit $\log T - \frac12\log L - x(\theta_0)$), estimate $g = (2\pi)^2 e^{-2\beta_0}$
  and give a 95 % confidence interval (transform the interval for $\beta_0$; mind the block coding). Compare it with
  $g = 9.81$ m/s². If 9.81 is outside the interval, which systematic error is the most likely cause? (Hint: an error of 1 cm
  in $L$ shifts the estimate of $g$ by about $1/L$ %.)

### 10. Reflection and conclusions

* With the $\hat\sigma$ from the main experiment, what was the actual power for the 10° vs 30° difference? Did the pilot
  estimate of $\sigma$ hold?
* Answer the three research questions in a few sentences.
* What would you change in the design if you had to repeat the experiment with the same amount of time?

### Optional bonus: comparing timing methods in a Latin square

Compare three timing methods (e.g. hand stopwatch, frame counting in a slow-motion video, a phone sensor app such as phyphox)
at one fixed setting of the pendulum. With three operators and three sessions, use a $3 \times 3$ Latin square
(rows = operators, columns = sessions, letters = methods; Lecture 4) or a replicated square, and analyse it. Which method is the
most precise (compare also the within-method variances)?

## Submission

* Teams of 2-4 students. One submission per team, by **pull request** into the `projects/` folder of the course repository
  (fork the repository, add your files, open a pull request; see `HW/README.md` for the steps):
  * `projects/01NAEX_Project1_<Surname1>_<Surname2>_....ipynb` - this notebook with all tasks answered and **with outputs**;
  * `projects/01NAEX_Project1_<Surname1>_<Surname2>_....csv` - the raw data, one row per run, with the columns
    `run, block, operator, date_time, L_m, theta_deg, mass_g, N, t_N_s, note` (and the pilot runs marked with `block = 0`).
* In the notebook, read the data from the raw URL of your fork, e.g.
  `https://raw.githubusercontent.com/<your-account>/01NAEX/main/projects/01NAEX_Project1_<Surnames>.csv`,
  so that the notebook runs in Google Colab.
* **Deadline: Sunday 8 November 2026, 23:59.**

### What is assessed

* the design and its justification (pilot study, sample size, blocking, randomization),
* the quality and documentation of the data (raw times, run order, notes),
* the correctness of the analysis (correct model and error term, diagnostics, equivalence test, estimate of $g$),
* the interpretation: conclusions supported by the data, honest discussion of the limitations,
* reproducibility: the notebook runs from top to bottom.